In [1]:
!pip install chromadb sentence-transformers

In [2]:
import chromadb
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')
client = chromadb.Client() #creates a local ChromaDB instance
collection = client.create_collection(name="documind_test") #consider this like a table in a database, it will hold all your data

chunks = [
    "The Eiffel Tower is located in Paris, France.",
    "Machine learning is a subset of artificial intelligence.",
    "Python is a popular programming language for data science.",
    "The Louvre museum contains thousands of art pieces.",
    "Neural networks are inspired by the human brain.",
    "Paris is the capital city of France.",
]
embeddings = model.encode(chunks).tolist()
collection.add(
    documents=chunks,
    embeddings=embeddings,
    ids=[f"chunk_{i}" for i in range(len(chunks))]
)
question = "Where is the Eiffel Tower?"
question_embedding = model.encode([question]).tolist()

results = collection.query(
    query_embeddings=question_embedding,
    n_results=3
)

c:\Users\Asus\anaconda3\envs\flask_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1732.32it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
for i, doc in enumerate(results['documents'][0]):
    print(f"Rank {i+1}: {doc}")

Rank 1: The Eiffel Tower is located in Paris, France.
Rank 2: Paris is the capital city of France.
Rank 3: The Louvre museum contains thousands of art pieces.


In [4]:
question2 = "Which city has the famous iron structure?"

question2_embedding = model.encode([question2]).tolist()
results2 = collection.query(
    query_embeddings=question2_embedding,
    n_results=2
)

print("\n\n🔍 Question 2:", question2)
print("(Notice: zero shared words with the answer)")
print("\n📄 Top 2 retrieved chunks:")
for i, doc in enumerate(results2['documents'][0]):
    print(f"\n  Rank {i+1}: {doc}")



🔍 Question 2: Which city has the famous iron structure?
(Notice: zero shared words with the answer)

📄 Top 2 retrieved chunks:

  Rank 1: The Eiffel Tower is located in Paris, France.

  Rank 2: Paris is the capital city of France.


In [5]:
!pip install langchain sentence-transformers langchain-community

In [8]:
import sys
!{sys.executable} -m pip install -U langchain-text-splitters

In [6]:
!pip install -U langchain langchain-text-splitters langchain-community chromadb sentence-transformers

In [2]:
# ============================================
# DAY 3 — Chunking Strategy Experiment
# ============================================

import sys
sys.path.append('../')  

from src.chunking.chunker import compare_strategies

sample_text = """
The Eiffel Tower was built in 1889 by Gustave Eiffel. 
It stands 330 meters tall and is located in Paris, France. 
The tower was originally criticized by French artists and intellectuals. 
Today it is the most visited monument in the world.

Machine learning is a subset of artificial intelligence. 
It enables systems to learn from data without being explicitly programmed. 
Neural networks are the backbone of modern deep learning systems. 
They are inspired by the structure of the human brain.

Python is the most popular language for data science. 
It has libraries like NumPy, Pandas, and Scikit-learn. 
These tools make data manipulation and model building efficient.
"""

compare_strategies(sample_text)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2198.14it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


CHUNKING STRATEGY COMPARISON

📦 Fixed Size: 4 chunks
----------------------------------------
Chunk 1 (199 chars):
The Eiffel Tower was built in 1889 by Gustave Eiffel. 
It stands 330 meters tall and is located in P...

Chunk 2 (200 chars):
s the most visited monument in the world.

Machine learning is a subset of artificial intelligence. ...

Chunk 3 (199 chars):
backbone of modern deep learning systems. 
They are inspired by the structure of the human brain.

P...


📦 Recursive: 2 chunks
----------------------------------------
Chunk 1 (497 chars):
The Eiffel Tower was built in 1889 by Gustave Eiffel. 
It stands 330 meters tall and is located in P...

Chunk 2 (175 chars):
Python is the most popular language for data science. 
It has libraries like NumPy, Pandas, and Scik...


📦 Sliding Window: 2 chunks
----------------------------------------
Chunk 1 (497 chars):
The Eiffel Tower was built in 1889 by Gustave Eiffel. 
It stands 330 meters tall and is located in P...

Chunk 2 (175 char

In [4]:
import sys
sys.path.append("../")

from src.chunking.chunker import (
    fixed_size_chunk,
    recursive_chunk,
    sliding_window_chunk,
    semantic_chunk
)

In [6]:
# Better output — see everything clearly
strategies = {
    "Fixed Size":     fixed_size_chunk(sample_text),
    "Recursive":      recursive_chunk(sample_text),
    "Sliding Window": sliding_window_chunk(sample_text),
    "Semantic":       semantic_chunk(sample_text)
}

for name, chunks in strategies.items():
    print(f"\n{'='*60}")
    print(f"📦 {name} — {len(chunks)} chunks total")
    print('='*60)

    for i, chunk in enumerate(chunks):
        print(f"\n  --- Chunk {i+1} ({len(chunk)} chars) ---")
        print(f"  {chunk}")


📦 Fixed Size — 4 chunks total

  --- Chunk 1 (199 chars) ---
  The Eiffel Tower was built in 1889 by Gustave Eiffel. 
It stands 330 meters tall and is located in Paris, France. 
The tower was originally criticized by French artists and intellectuals. 
Today it i

  --- Chunk 2 (200 chars) ---
  s the most visited monument in the world.

Machine learning is a subset of artificial intelligence. 
It enables systems to learn from data without being explicitly programmed. 
Neural networks are the

  --- Chunk 3 (199 chars) ---
  backbone of modern deep learning systems. 
They are inspired by the structure of the human brain.

Python is the most popular language for data science. 
It has libraries like NumPy, Pandas, and Scik

  --- Chunk 4 (75 chars) ---
  it-learn. 
These tools make data manipulation and model building efficient.

📦 Recursive — 2 chunks total

  --- Chunk 1 (497 chars) ---
  The Eiffel Tower was built in 1889 by Gustave Eiffel. 
It stands 330 meters tall and is located in

In [ ]:
# ============================================
# EXPERIMENT 1 — Fixed Size vs Recursive
# See exactly where Fixed Size breaks
# ============================================

import sys
sys.path.append('../')
from src.chunking.chunker import fixed_size_chunk, recursive_chunk

sample_text = """
The Eiffel Tower was built in 1889 by Gustave Eiffel. 
It stands 330 meters tall and is located in Paris, France. 
The tower was originally criticized by French artists and intellectuals. 
Today it is the most visited monument in the world.

Machine learning is a subset of artificial intelligence. 
It enables systems to learn from data without being explicitly programmed. 
Neural networks are the backbone of modern deep learning systems. 
They are inspired by the structure of the human brain.

Python is the most popular language for data science. 
It has libraries like NumPy, Pandas, and Scikit-learn. 
These tools make data manipulation and model building efficient.
"""

# --- Fixed Size ---
fixed_chunks = fixed_size_chunk(sample_text)
print("📦 FIXED SIZE CHUNKS")
print("="*60)
for i, chunk in enumerate(fixed_chunks):
    print(f"\nChunk {i+1} ({len(chunk)} chars):")
    print(chunk)
    print("-"*40)


📦 FIXED SIZE CHUNKS

Chunk 1 (199 chars):
The Eiffel Tower was built in 1889 by Gustave Eiffel. 
It stands 330 meters tall and is located in Paris, France. 
The tower was originally criticized by French artists and intellectuals. 
Today it i
----------------------------------------

Chunk 2 (200 chars):
s the most visited monument in the world.

Machine learning is a subset of artificial intelligence. 
It enables systems to learn from data without being explicitly programmed. 
Neural networks are the
----------------------------------------

Chunk 3 (199 chars):
backbone of modern deep learning systems. 
They are inspired by the structure of the human brain.

Python is the most popular language for data science. 
It has libraries like NumPy, Pandas, and Scik
----------------------------------------

Chunk 4 (75 chars):
it-learn. 
These tools make data manipulation and model building efficient.
----------------------------------------


📦 RECURSIVE CHUNKS

Chunk 1 (497 chars):
The Eiffe

In [8]:

# --- Recursive ---
recursive_chunks = recursive_chunk(sample_text)
print("\n\n📦 RECURSIVE CHUNKS")
print("="*60)
for i, chunk in enumerate(recursive_chunks):
    print(f"\nChunk {i+1} ({len(chunk)} chars):")
    print(chunk)
    print("-"*40)



📦 RECURSIVE CHUNKS

Chunk 1 (497 chars):
The Eiffel Tower was built in 1889 by Gustave Eiffel. 
It stands 330 meters tall and is located in Paris, France. 
The tower was originally criticized by French artists and intellectuals. 
Today it is the most visited monument in the world.

Machine learning is a subset of artificial intelligence. 
It enables systems to learn from data without being explicitly programmed. 
Neural networks are the backbone of modern deep learning systems. 
They are inspired by the structure of the human brain.
----------------------------------------

Chunk 2 (175 chars):
Python is the most popular language for data science. 
It has libraries like NumPy, Pandas, and Scikit-learn. 
These tools make data manipulation and model building efficient.
----------------------------------------


In [9]:
print("=== FIXED SIZE ===")
for i, chunk in enumerate(fixed_size_chunk(sample_text)):
    print(f"\nChunk {i+1}:\n{chunk}")

print("\n=== RECURSIVE ===")
for i, chunk in enumerate(recursive_chunk(sample_text)):
    print(f"\nChunk {i+1}:\n{chunk}")

=== FIXED SIZE ===

Chunk 1:
The Eiffel Tower was built in 1889 by Gustave Eiffel. 
It stands 330 meters tall and is located in Paris, France. 
The tower was originally criticized by French artists and intellectuals. 
Today it i

Chunk 2:
s the most visited monument in the world.

Machine learning is a subset of artificial intelligence. 
It enables systems to learn from data without being explicitly programmed. 
Neural networks are the

Chunk 3:
backbone of modern deep learning systems. 
They are inspired by the structure of the human brain.

Python is the most popular language for data science. 
It has libraries like NumPy, Pandas, and Scik

Chunk 4:
it-learn. 
These tools make data manipulation and model building efficient.

=== RECURSIVE ===

Chunk 1:
The Eiffel Tower was built in 1889 by Gustave Eiffel. 
It stands 330 meters tall and is located in Paris, France. 
The tower was originally criticized by French artists and intellectuals. 
Today it is the most visited monument in the w

In [10]:
print("=== SLIDING WINDOW ===")
for i, chunk in enumerate(sliding_window_chunk(sample_text)):
    print(f"\nChunk {i+1}:\n{chunk}")
    print("-"*40)

=== SLIDING WINDOW ===

Chunk 1:
The Eiffel Tower was built in 1889 by Gustave Eiffel. 
It stands 330 meters tall and is located in Paris, France. 
The tower was originally criticized by French artists and intellectuals. 
Today it is the most visited monument in the world.

Machine learning is a subset of artificial intelligence. 
It enables systems to learn from data without being explicitly programmed. 
Neural networks are the backbone of modern deep learning systems. 
They are inspired by the structure of the human brain.
----------------------------------------

Chunk 2:
Python is the most popular language for data science. 
It has libraries like NumPy, Pandas, and Scikit-learn. 
These tools make data manipulation and model building efficient.
----------------------------------------


In [11]:
print("=== SEMANTIC CHUNKING ===")
semantic_chunks = semantic_chunk(sample_text)
for i, chunk in enumerate(semantic_chunks):
    print(f"\nChunk {i+1}:\n{chunk}")
    print("-"*40)

=== SEMANTIC CHUNKING ===

Chunk 1:
The Eiffel Tower was built in 1889 by Gustave Eiffel. It stands 330 meters tall and is located in Paris, France.
----------------------------------------

Chunk 2:
The tower was originally criticized by French artists and intellectuals.
----------------------------------------

Chunk 3:
Today it is the most visited monument in the world.
----------------------------------------

Chunk 4:
Machine learning is a subset of artificial intelligence. It enables systems to learn from data without being explicitly programmed. Neural networks are the backbone of modern deep learning systems. They are inspired by the structure of the human brain.
----------------------------------------

Chunk 5:
Python is the most popular language for data science. It has libraries like NumPy, Pandas, and Scikit-learn. These tools make data manipulation and model building efficient.
----------------------------------------


In [12]:
# More aggressive splitting (lower threshold = splits more easily)
chunks_low = semantic_chunk(sample_text, threshold=0.1)
print(f"Threshold 0.1: {len(chunks_low)} chunks")

# Less aggressive splitting (higher threshold = stays together more)
chunks_high = semantic_chunk(sample_text, threshold=0.6)
print(f"Threshold 0.6: {len(chunks_high)} chunks")

# Print high threshold result
for i, chunk in enumerate(chunks_high):
    print(f"\nChunk {i+1}:\n{chunk}")
    print("-"*40)

Threshold 0.1: 3 chunks
Threshold 0.6: 11 chunks

Chunk 1:
The Eiffel Tower was built in 1889 by Gustave Eiffel.
----------------------------------------

Chunk 2:
It stands 330 meters tall and is located in Paris, France.
----------------------------------------

Chunk 3:
The tower was originally criticized by French artists and intellectuals.
----------------------------------------

Chunk 4:
Today it is the most visited monument in the world.
----------------------------------------

Chunk 5:
Machine learning is a subset of artificial intelligence.
----------------------------------------

Chunk 6:
It enables systems to learn from data without being explicitly programmed.
----------------------------------------

Chunk 7:
Neural networks are the backbone of modern deep learning systems.
----------------------------------------

Chunk 8:
They are inspired by the structure of the human brain.
----------------------------------------

Chunk 9:
Python is the most popular language for da